# Search-6-AdversarialSearch : Recherche Adversariale

**Navigation** : [<< Recherche informee](Search-3-Informed.ipynb) | [Index](../README.md) | [MCTS >>](Search-7-MCTS-And-Beyond.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Modeliser** un jeu a somme nulle comme un probleme de recherche
2. **Implementer** l'algorithme Minimax avec l'optimisation Alpha-Beta
3. **Comprendre** les limites de Minimax et les strategies d'amelioration
4. **Appliquer** la recherche iterative (iterative deepening)
5. **Utiliser** les tables de transposition pour optimiser la recherche

### Prerequis
- Notebooks Search-1 (StateSpace) et Search-2 (DFS/BFS)
- Notebook Search-3 (heuristiques)
- Bases de Python : recursivite, classes

### Duree estimee : 60 minutes

In [ ]:
# Imports
import sys
import time
import random
from typing import Optional, List, Dict, Tuple, Any, Callable
from copy import deepcopy
from functools import lru_cache

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

print("Environnement pret pour la recherche adversariale.")

## 1. Introduction : Jeux a Somme Nulle

### Qu'est-ce qu'un jeu a somme nulle ?

Un **jeu a somme nulle** (zero-sum game) est un jeu ou les gains d'un joueur correspondent exactement aux pertes de l'autre. Le gain total des deux joueurs est toujours zero.

**Exemples classiques :**
- Tic-Tac-Toe (Morpion)
- Echecs
- Go
- Puissance 4 (Connect Four)

### Caracteristiques d'un jeu a information parfaite

| Caracteristique | Description |
|----------------|-------------|
| **Information parfaite** | Les deux joueurs connaissent l'etat complet du jeu |
| **Tour a tour** | Les joueurs jouent alternativement |
| **Deterministe** | Le resultat d'une action est certain (pas de hasard) |
| **Fini** | Le jeu se termine toujours en un nombre fini de coups |

### Formalisation

Un jeu a somme nulle peut etre modelise comme :

- **S** : Ensemble des etats du jeu (positions)
- **s0** : Etat initial
- **Joueur(s)** : Fonction indiquant quel joueur doit jouer dans l'etat s
- **Actions(s)** : Coups legaux dans l'etat s
- **Resultat(s, a)** : Nouvel etat apres avoir joue l'action a
- **EstTerminal(s)** : Le jeu est-il termine ?
- **Utilite(s, p)** : Valeur finale pour le joueur p (-1, 0, +1)

In [ ]:
# Classe de base abstraite pour un jeu a somme nulle
from abc import ABC, abstractmethod

class JeuSommeNulle(ABC):
    """Interface pour un jeu a somme nulle a information parfaite."""
    
    @abstractmethod
    def etat_initial(self) -> Any:
        """Retourne l'etat initial du jeu."""
        pass
    
    @abstractmethod
    def joueur(self, etat: Any) -> str:
        """Retourne le joueur qui doit jouer ('MAX' ou 'MIN')."""
        pass
    
    @abstractmethod
    def actions(self, etat: Any) -> List[Any]:
        """Retourne la liste des actions legales."""
        pass
    
    @abstractmethod
    def resultat(self, etat: Any, action: Any) -> Any:
        """Retourne le nouvel etat apres l'action."""
        pass
    
    @abstractmethod
    def est_terminal(self, etat: Any) -> bool:
        """Le jeu est-il termine ?"""
        pass
    
    @abstractmethod
    def utilite(self, etat: Any, joueur: str) -> float:
        """Valeur de l'etat terminal pour le joueur (+1, 0, -1)."""
        pass
    
    @abstractmethod
    def afficher(self, etat: Any) -> str:
        """Representation textuelle de l'etat."""
        pass

## 2. Exemple : Tic-Tac-Toe (Morpion)

Implementons le jeu de Morpour pour illustrer les concepts.

In [ ]:
class TicTacToe(JeuSommeNulle):
    """Implementation du jeu de Morpion."""
    
    def __init__(self):
        self._etat_initial = (tuple([' ']*9), 'X')  # (grille 3x3 aplatie, joueur)
    
    def etat_initial(self) -> Tuple[Tuple[str, ...], str]:
        return self._etat_initial
    
    def joueur(self, etat: Tuple[Tuple[str, ...], str]) -> str:
        return 'MAX' if etat[1] == 'X' else 'MIN'
    
    def actions(self, etat: Tuple[Tuple[str, ...], str]) -> List[int]:
        """Retourne les indices des cases vides."""
        grille = etat[0]
        return [i for i in range(9) if grille[i] == ' ']
    
    def resultat(self, etat: Tuple[Tuple[str, ...], str], action: int) -> Tuple[Tuple[str, ...], str]:
        """Joue le coup et retourne le nouvel etat."""
        grille = list(etat[0])
        joueur_actuel = etat[1]
        grille[action] = joueur_actuel
        prochain_joueur = 'O' if joueur_actuel == 'X' else 'X'
        return (tuple(grille), prochain_joueur)
    
    def est_terminal(self, etat: Tuple[Tuple[str, ...], str]) -> bool:
        grille = etat[0]
        lignes = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],  # lignes
            [0, 3, 6], [1, 4, 7], [2, 5, 8],  # colonnes
            [0, 4, 8], [2, 4, 6]              # diagonales
        ]
        for l in lignes:
            if grille[l[0]] != ' ' and grille[l[0]] == grille[l[1]] == grille[l[2]]:
                return True
        return ' ' not in grille
    
    def utilite(self, etat: Tuple[Tuple[str, ...], str], joueur: str) -> float:
        grille = etat[0]
        lignes = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],
            [0, 3, 6], [1, 4, 7], [2, 5, 8],
            [0, 4, 8], [2, 4, 6]
        ]
        for l in lignes:
            if grille[l[0]] != ' ' and grille[l[0]] == grille[l[1]] == grille[l[2]]:
                gagnant = 'MAX' if grille[l[0]] == 'X' else 'MIN'
                return 1 if gagnant == joueur else -1
        return 0
    
    def afficher(self, etat: Tuple[Tuple[str, ...], str]) -> str:
        g = etat[0]
        return f"\n{g[0]}|{g[1]}|{g[2]}\n-----\n{g[3]}|{g[4]}|{g[5]}\n-----\n{g[6]}|{g[7]}|{g[8]}\n"

# Test
jeu = TicTacToe()
print(jeu.afficher(jeu.etat_initial()))
print(f"Joueur actuel: {jeu.joueur(jeu.etat_initial())}")
print(f"Actions possibles: {jeu.actions(jeu.etat_initial())}")

## 3. L'Algorithme Minimax

### Principe

L'algorithme **Minimax** explore l'arbre de jeu complet jusqu'aux etats terminaux :

- **MAX** cherche a maximiser l'utilite (son tour)
- **MIN** cherche a minimiser l'utilite (tour adverse)

### Pseudo-code

```
fonction MINIMAX(etat):
    si EST_TERMINAL(etat):
        retourner UTILITE(etat)
    
    si JOUEUR(etat) == MAX:
        retourner max(MINIMAX(RESULTAT(etat, a)) pour a dans ACTIONS(etat))
    sinon:
        retourner min(MINIMAX(RESULTAT(etat, a)) pour a dans ACTIONS(etat))
```

In [ ]:
def minimax(jeu: JeuSommeNulle, etat: Any, joueur_max: str = 'MAX') -> Tuple[float, Optional[Any]]:
    """
    Algorithme Minimax recursif.
    Retourne (valeur, meilleure_action)
    """
    if jeu.est_terminal(etat):
        return jeu.utilite(etat, joueur_max), None
    
    actions_legales = jeu.actions(etat)
    
    if jeu.joueur(etat) == joueur_max:
        meilleure_valeur = float('-inf')
        meilleure_action = None
        
        for action in actions_legales:
            nouvel_etat = jeu.resultat(etat, action)
            valeur, _ = minimax(jeu, nouvel_etat, joueur_max)
            if valeur > meilleure_valeur:
                meilleure_valeur = valeur
                meilleure_action = action
        return meilleure_valeur, meilleure_action
    else:
        meilleure_valeur = float('+inf')
        meilleure_action = None
        
        for action in actions_legales:
            nouvel_etat = jeu.resultat(etat, action)
            valeur, _ = minimax(jeu, nouvel_etat, joueur_max)
            if valeur < meilleure_valeur:
                meilleure_valeur = valeur
                meilleure_action = action
        return meilleure_valeur, meilleure_action

# Test sur Tic-Tac-Toe
jeu = TicTacToe()
valeur, action = minimax(jeu, jeu.etat_initial())
print(f"Valeur Minimax depuis l'etat initial: {valeur}")
print(f"Meilleure action: {action}")

### Analyse de complexite

| Metrique | Valeur |
|----------|--------|
| **Complexite temporelle** | O(b^m) ou b = branching factor, m = profondeur max |
| **Complexite spatiale** | O(m) pour la pile de recursion |

Pour le Tic-Tac-Toe : b = 9, m = 9, environ 9! = 362,880 noeuds.

Pour les echecs : b ~ 35, m ~ 100, environ 35^100 noeuds (impossible !)

## 4. L'Elagage Alpha-Beta

### Principe

L'elagage **Alpha-Beta** permet d'eliminer des branches entieres de l'arbre sans changer le resultat.

- **alpha** : meilleure valeur que MAX peut garantir
- **beta** : meilleure valeur que MIN peut garantir

Si alpha >= beta, on peut couper la branche.

### Gain de performance

Avec un ordonnancement optimal : O(b^(m/2)) au lieu de O(b^m).

In [ ]:
def alpha_beta(jeu: JeuSommeNulle, etat: Any, alpha: float = float('-inf'), 
               beta: float = float('+inf'), joueur_max: str = 'MAX') -> Tuple[float, Optional[Any]]:
    """
    Algorithme Alpha-Beta pruning.
    """
    if jeu.est_terminal(etat):
        return jeu.utilite(etat, joueur_max), None
    
    actions_legales = jeu.actions(etat)
    
    if jeu.joueur(etat) == joueur_max:
        meilleure_valeur = float('-inf')
        meilleure_action = None
        
        for action in actions_legales:
            nouvel_etat = jeu.resultat(etat, action)
            valeur, _ = alpha_beta(jeu, nouvel_etat, alpha, beta, joueur_max)
            if valeur > meilleure_valeur:
                meilleure_valeur = valeur
                meilleure_action = action
            alpha = max(alpha, meilleure_valeur)
            if beta <= alpha:
                break
        return meilleure_valeur, meilleure_action
    else:
        meilleure_valeur = float('+inf')
        meilleure_action = None
        
        for action in actions_legales:
            nouvel_etat = jeu.resultat(etat, action)
            valeur, _ = alpha_beta(jeu, nouvel_etat, alpha, beta, joueur_max)
            if valeur < meilleure_valeur:
                meilleure_valeur = valeur
                meilleure_action = action
            beta = min(beta, meilleure_valeur)
            if beta <= alpha:
                break
        return meilleure_valeur, meilleure_action

# Benchmark
jeu = TicTacToe()
start = time.time()
v1, a1 = minimax(jeu, jeu.etat_initial())
t1 = time.time() - start

start = time.time()
v2, a2 = alpha_beta(jeu, jeu.etat_initial())
t2 = time.time() - start

print(f"Minimax: valeur={v1}, temps={t1:.4f}s")
print(f"Alpha-Beta: valeur={v2}, temps={t2:.4f}s")
print(f"Speedup: {t1/t2:.1f}x")

## 5. Recherche Iterative (Iterative Deepening)

Explore progressivement en augmentant la profondeur. Permet de controler le temps et d'utiliser les resultats des profondeurs precedentes pour ordonner les coups.

In [ ]:
def evaluation_heuristique(jeu: JeuSommeNulle, etat: Any, joueur: str) -> float:
    """Fonction d'evaluation pour Tic-Tac-Toe."""
    grille = etat[0]
    mon_symbole = 'X' if joueur == 'MAX' else 'O'
    adv_symbole = 'O' if joueur == 'MAX' else 'X'
    
    lignes = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],
        [0, 3, 6], [1, 4, 7], [2, 5, 8],
        [0, 4, 8], [2, 4, 6]
    ]
    
    score = 0
    for l in lignes:
        ma_ligne = sum(1 for i in l if grille[i] == mon_symbole)
        adv_ligne = sum(1 for i in l if grille[i] == adv_symbole)
        if adv_ligne == 0:
            score += ma_ligne ** 2
        if ma_ligne == 0:
            score -= adv_ligne ** 2
    return score / 9.0

def alpha_beta_limite(jeu, etat, profondeur, alpha, beta, joueur_max='MAX'):
    """Alpha-Beta avec profondeur limitee."""
    if jeu.est_terminal(etat):
        return jeu.utilite(etat, joueur_max), None
    if profondeur == 0:
        return evaluation_heuristique(jeu, etat, joueur_max), None
    
    actions_legales = jeu.actions(etat)
    if jeu.joueur(etat) == joueur_max:
        best_v, best_a = float('-inf'), None
        for action in actions_legales:
            v, _ = alpha_beta_limite(jeu, jeu.resultat(etat, action), profondeur-1, alpha, beta, joueur_max)
            if v > best_v:
                best_v, best_a = v, action
            alpha = max(alpha, best_v)
            if beta <= alpha:
                break
        return best_v, best_a
    else:
        best_v, best_a = float('+inf'), None
        for action in actions_legales:
            v, _ = alpha_beta_limite(jeu, jeu.resultat(etat, action), profondeur-1, alpha, beta, joueur_max)
            if v < best_v:
                best_v, best_a = v, action
            beta = min(beta, best_v)
            if beta <= alpha:
                break
        return best_v, best_a

def iterative_deepening(jeu, etat, temps_max=1.0):
    """Recherche iterative deepening avec limite de temps."""
    start = time.time()
    best_action = None
    best_value = 0
    depth = 1
    
    while time.time() - start < temps_max:
        value, action = alpha_beta_limite(jeu, etat, depth, float('-inf'), float('+inf'))
        best_value, best_action = value, action
        if abs(value) >= 1:  # Victoire certaine
            break
        depth += 1
    
    return best_value, best_action, depth

# Test
v, a, d = iterative_deepening(jeu, jeu.etat_initial(), temps_max=0.5)
print(f"Iterative Deepening: valeur={v:.2f}, action={a}, profondeur={d}")

## 6. Tables de Transposition

Les **tables de transposition** stockent les resultats des etats deja evalues. Differentes sequences de coups peuvent mener au meme etat (transpositions).

In [ ]:
class AlphaBetaTransposition:
    """Alpha-Beta avec table de transposition."""
    
    def __init__(self):
        self.table = {}
        self.stats = {'hits': 0, 'misses': 0}
    
    def rechercher(self, jeu, etat, profondeur, alpha, beta, joueur_max='MAX'):
        h = hash(etat)
        
        if h in self.table:
            cached_v, cached_d, flag = self.table[h]
            if cached_d >= profondeur:
                self.stats['hits'] += 1
                if flag == 'exact':
                    return cached_v, None
        
        self.stats['misses'] += 1
        
        if jeu.est_terminal(etat):
            return jeu.utilite(etat, joueur_max), None
        if profondeur == 0:
            return evaluation_heuristique(jeu, etat, joueur_max), None
        
        actions = jeu.actions(etat)
        if jeu.joueur(etat) == joueur_max:
            best_v, best_a = float('-inf'), None
            for action in actions:
                v, _ = self.rechercher(jeu, jeu.resultat(etat, action), profondeur-1, alpha, beta, joueur_max)
                if v > best_v:
                    best_v, best_a = v, action
                alpha = max(alpha, best_v)
                if beta <= alpha:
                    break
            self.table[h] = (best_v, profondeur, 'exact')
            return best_v, best_a
        else:
            best_v, best_a = float('+inf'), None
            for action in actions:
                v, _ = self.rechercher(jeu, jeu.resultat(etat, action), profondeur-1, alpha, beta, joueur_max)
                if v < best_v:
                    best_v, best_a = v, action
                beta = min(beta, best_v)
                if beta <= alpha:
                    break
            self.table[h] = (best_v, profondeur, 'exact')
            return best_v, best_a

# Test
ab_trans = AlphaBetaTransposition()
start = time.time()
v, a = ab_trans.rechercher(jeu, jeu.etat_initial(), 9, float('-inf'), float('+inf'))
t = time.time() - start

print(f"Alpha-Beta + Transposition: valeur={v}, temps={t:.4f}s")
print(f"Cache: {ab_trans.stats['hits']} hits, {ab_trans.stats['misses']} misses")

## 7. Benchmark Comparatif

In [ ]:
# Benchmark complet
jeu = TicTacToe()
resultats = []

# Test Minimax
start = time.time()
v1, a1 = minimax(jeu, jeu.etat_initial())
t1 = time.time() - start
resultats.append(('Minimax', t1, v1))

# Test Alpha-Beta
start = time.time()
v2, a2 = alpha_beta(jeu, jeu.etat_initial())
t2 = time.time() - start
resultats.append(('Alpha-Beta', t2, v2))

# Test Alpha-Beta + Transposition
ab_trans = AlphaBetaTransposition()
start = time.time()
v3, a3 = ab_trans.rechercher(jeu, jeu.etat_initial(), 9, float('-inf'), float('+inf'))
t3 = time.time() - start
resultats.append(('Alpha-Beta + Trans', t3, v3))

# Affichage
df = pd.DataFrame(resultats, columns=['Algorithme', 'Temps (s)', 'Valeur'])
df['Speedup'] = df['Temps (s)'].apply(lambda x: f"{t1/x:.1f}x")
display(df)

# Graphique
fig, ax = plt.subplots(figsize=(10, 5))
algos = [r[0] for r in resultats]
temps = [r[1] for r in resultats]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']

bars = ax.bar(algos, temps, color=colors, edgecolor='black')
ax.set_ylabel('Temps (secondes)')
ax.set_title('Comparaison des algorithmes de recherche adversariale')
ax.set_yscale('log')

for bar, t in zip(bars, temps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{t:.4f}s',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Synthese

### Resume des techniques

| Technique | Gain | Complexite |
|-----------|------|------------|
| **Minimax** | Base | O(b^m) |
| **Alpha-Beta** | 2x-10x | O(b^(m/2)) optimal |
| **Transposition Tables** | 2x-5x | Memoire O(n) |
| **Iterative Deepening** | Controle temps | Surcout negligeable |

### Limites

1. **Explosion combinatoire** : Profondeur restreinte
2. **Horizon effect** : Decisions cachees au-dela de la profondeur
3. **Fonction d'evaluation** : Qualite depend de l'heuristique

### Pour aller plus loin

- **MCTS** (Monte Carlo Tree Search) : Explorer intelligemment sans fonction d'evaluation
- **Reseaux de neurones** : AlphaGo, AlphaZero

---

**Navigation** : [<< Recherche informee](Search-3-Informed.ipynb) | [Index](../README.md) | [MCTS >>](Search-7-MCTS-And-Beyond.ipynb)

## Exercices

1. **Connect Four** : Implementez une classe Connect Four et testez Alpha-Beta
2. **Ordonnancement** : Ameliorez Alpha-Beta en ordonnant les coups par potentiel
3. **Quiescence Search** : Implementez une recherche de quiescence
4. **Zobrist Hashing** : Implementez le Zobrist hashing pour Connect Four